# 02 — Clean, feature-build, and merge

**Inputs:** `data/uscis_*.csv` (7 files), `data/lca_slim/lca_*.csv` (15 files)  
**Function:** construct one canonical employer–fiscal-year record, build strictly
backward-looking sponsorship features, aggregate DOL records to the same grain, and join
the agencies  
**Outputs:** `data/analysis_panel.csv`, `data/lca_employer_year.csv`,
`data/analysis_panel_wages.csv`

This notebook performs four joins. Every join prints both input tables immediately before
the operation and verifies row count, key uniqueness, and match status immediately after.

In [1]:
import sys
from pathlib import Path

# Locate the repository root by walking up until code/src is found, then put the
# code directory on the path. No absolute paths, so this runs from any checkout.
_here = Path.cwd().resolve()
_root = next(p for p in (_here, *_here.parents) if (p / "code" / "src").is_dir())
sys.path.insert(0, str(_root / "code"))

import numpy as np
import pandas as pd

from src import (DATA, LCA_SLIM, TABLES, USCIS_YEARS, MODEL_YEARS,
                 canon, annualise, clean_wage_level,
                 describe_frame, merge_report)
from src.wages import WAGE_MIN, WAGE_MAX

## Functions

All reusable logic is defined before the pipeline is executed.

In [2]:
RENAME = {
    "Initial Approval": "Initial Approvals", "Initial Denial": "Initial Denials",
    "Continuing Approval": "Continuing Approvals",
    "Continuing Denial": "Continuing Denials",
}

NAICS_LABEL = {
    "11": "Agriculture", "21": "Mining", "22": "Utilities", "23": "Construction",
    "31": "Manufacturing", "32": "Manufacturing", "33": "Manufacturing",
    "42": "Wholesale trade", "44": "Retail trade", "45": "Retail trade",
    "48": "Transportation", "49": "Transportation", "51": "Information",
    "52": "Finance/insurance", "53": "Real estate",
    "54": "Professional/scientific/technical", "55": "Management of companies",
    "56": "Administrative/support", "61": "Educational services",
    "62": "Health care/social assistance", "71": "Arts/entertainment",
    "72": "Accommodation/food", "81": "Other services",
    "92": "Public administration", "99": "Unknown",
}

COUNTS = ["initial_approvals", "initial_denials",
          "continuing_approvals", "continuing_denials"]


def load_uscis():
    '''Stack annual USCIS files, reconcile headers, and preserve every count.'''
    frames = []
    for fy in USCIS_YEARS:
        d = pd.read_csv(DATA / f"uscis_{fy}.csv", dtype=str).rename(columns=RENAME)
        d["source_file_fy"] = fy
        frames.append(d)
    raw = pd.concat(frames, ignore_index=True)
    raw["source_row"] = np.arange(len(raw))

    for c in ["Fiscal Year", "Initial Approvals", "Initial Denials",
              "Continuing Approvals", "Continuing Denials"]:
        raw[c] = pd.to_numeric(
            raw[c].astype(str).str.replace(",", "", regex=False), errors="coerce")
    raw.columns = [c.lower().replace(" ", "_") for c in raw.columns]
    raw = raw.rename(columns={"fiscal_year": "fy"})
    raw[COUNTS] = raw[COUNTS].fillna(0)
    raw["fy"] = raw.fy.fillna(raw.source_file_fy).astype(int)
    return raw


def prepare_uscis(raw):
    '''Create canonical keys and descriptor fields before employer-year aggregation.'''
    d = raw.copy()
    employer = d.employer.fillna("").astype(str).str.strip().str.upper()
    missing = employer.eq("")
    employer.loc[missing] = "UNKNOWN EMPLOYER ROW " + d.loc[missing, "source_row"].astype(str)
    d["employer"] = employer
    d["key"] = d.employer.map(canon)
    blank_key = d.key.eq("")
    d.loc[blank_key, "key"] = "UNKNOWN EMPLOYER KEY " + d.loc[blank_key, "source_row"].astype(str)
    d["state"] = d.state.fillna("UNKNOWN").astype(str).str.strip().str.upper()
    d["naics"] = (d.naics.fillna("99").astype(str)
                  .str.extract(r"(\d{2})", expand=False).fillna("99"))
    d["initial_total"] = d.initial_approvals + d.initial_denials
    d["continuing_total"] = d.continuing_approvals + d.continuing_denials
    d["petitions_total"] = d.initial_total + d.continuing_total
    return d


def history_table(panel):
    '''Shift fiscal year t records to t+1 so all history is known before the outcome.'''
    hist = panel[["key", "fy", "initial_total", "initial_denials",
                  "petitions_total"]].copy()
    hist["fy"] = hist.fy + 1
    hist = hist.rename(columns={"initial_total": "prev_initial",
                                "initial_denials": "prev_denials",
                                "petitions_total": "prev_petitions"})
    hist["prev_denial_rate"] = (
        hist.prev_denials / hist.prev_initial.replace(0, np.nan))
    return hist


def yes_no(series):
    '''Map explicit yes/no labels while preserving truly missing values.'''
    s = series.fillna("").astype(str).str.upper().str.strip()
    yes = s.isin(["Y", "YES", "1", "TRUE"])
    no = s.isin(["N", "NO", "0", "FALSE"])
    return pd.Series(np.select([yes, no], [1.0, 0.0], default=np.nan), index=series.index)


def load_lca():
    '''Load slim LCA extracts, retain certified H-1B filings, and clean wages.'''
    frames = []
    for p in sorted(LCA_SLIM.glob("lca_*.csv")):
        d = pd.read_csv(p, dtype=str, low_memory=False)
        d["fy"] = int(p.stem.split("_")[1])
        frames.append(d)
    if not frames:
        raise FileNotFoundError("No data/lca_slim/lca_*.csv files were found")
    lca = pd.concat(frames, ignore_index=True)
    print(f"BEFORE filtering: {len(lca):,} extracted rows")

    lca["case_status"] = lca.case_status.fillna("").str.upper().str.strip()
    lca["visa_class"] = lca.visa_class.fillna("").str.upper().str.strip()
    lca = lca[lca.visa_class.eq("H-1B")].copy()
    print(f"    after H-1B filter:      {len(lca):,}")
    lca = lca[lca.case_status.isin(
        ["CERTIFIED", "CERTIFIED-WITHDRAWN", "CERTIFIED - WITHDRAWN"])].copy()
    print(f"    after certified filter: {len(lca):,}")

    for c in ["wage_from", "prevailing_wage", "n_workers"]:
        lca[c] = pd.to_numeric(
            lca[c].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce")
    lca["position_weight"] = lca.n_workers.where(lca.n_workers > 0, 1).fillna(1)
    lca["wage_annual"] = annualise(lca.wage_from, lca.wage_unit)
    lca["pw_annual"] = annualise(lca.prevailing_wage, lca.pw_unit)
    ok = (lca.wage_annual.between(WAGE_MIN, WAGE_MAX) &
          lca.pw_annual.between(WAGE_MIN, WAGE_MAX))
    print(f"    wages inside [{WAGE_MIN:,}, {WAGE_MAX:,}]: {ok.mean():.1%}")
    lca["wage_ratio"] = np.where(ok, lca.wage_annual / lca.pw_annual, np.nan)
    lca.loc[~ok, ["wage_annual", "pw_annual"]] = np.nan

    level = clean_wage_level(lca.pw_level)
    lca["is_low_level"] = np.where(level.isin(["I", "II"]), 1.0,
                                    np.where(level.isin(["III", "IV"]), 0.0, np.nan))
    lca["low_level_positions"] = lca.is_low_level * lca.position_weight
    lca["known_level_positions"] = np.where(
        lca.is_low_level.notna(), lca.position_weight, 0.0)
    lca["soc2"] = lca.soc_code.astype(str).str.extract(r"^(\d{2})")[0]
    lca["dep"] = yes_no(lca.h1b_dependent)
    lca["willful"] = yes_no(lca.willful_violator)
    lca["agent"] = yes_no(lca.agent_used)
    lca["ft"] = yes_no(lca.full_time)
    lca["key"] = lca.employer_name.map(canon)
    missing_key = lca.key.eq("")
    print(f"    blank employer keys dropped: {missing_key.sum():,}")
    return lca.loc[~missing_key].copy()

## 1. Construct the USCIS employer-year panel

USCIS rows are employer-by-worksite cells. The inferential unit must instead be one
canonical employer in one fiscal year. Counts are summed across worksites, while the
state and industry of the worksite with the most petitions supply descriptive labels.
Missing employer names receive unique placeholders so that aggregation cannot silently
discard their petition counts.

### Merge 1 of 4: totals plus dominant worksite descriptors

In [3]:
raw = load_uscis()
prepared = prepare_uscis(raw)
describe_frame(prepared, "BEFORE merge 1: prepared USCIS worksite rows",
               keys=["key", "fy"])

totals = prepared.groupby(["key", "fy"], as_index=False)[COUNTS].sum()
totals["initial_total"] = totals.initial_approvals + totals.initial_denials
totals["continuing_total"] = totals.continuing_approvals + totals.continuing_denials
totals["petitions_total"] = totals.initial_total + totals.continuing_total

dominant = (prepared.sort_values(
    ["key", "fy", "petitions_total", "initial_total", "source_row"],
    ascending=[True, True, False, False, True])
    .drop_duplicates(["key", "fy"])
    [["key", "fy", "employer", "naics", "state"]])

describe_frame(totals, "BEFORE merge 1 left: summed petition counts", keys=["key", "fy"])
describe_frame(dominant, "BEFORE merge 1 right: dominant descriptors", keys=["key", "fy"])
panel_all = totals.merge(dominant, on=["key", "fy"], how="left", validate="one_to_one")
describe_frame(panel_all, "AFTER merge 1: canonical employer-years", keys=["key", "fy"])
print(f"    rows: {len(totals):,} -> {len(panel_all):,}")

print("\nPetition conservation from raw USCIS rows to canonical employer-years:")
for c in COUNTS:
    before, after = prepared[c].sum(), panel_all[c].sum()
    status = "OK" if np.isclose(before, after) else "MISMATCH"
    print(f"    {c:24s} {before:>10,.0f} -> {after:>10,.0f}  {status}")
    assert np.isclose(before, after), f"petition conservation failed for {c}"
assert not panel_all.duplicated(["key", "fy"]).any()

--- BEFORE merge 1: prepared USCIS worksite rows ---
    rows: 374,253   columns: 17
    distinct key: 153,165
    distinct fy: 7
    columns with missing values: 3 (worst: zip at 0.7%)


--- BEFORE merge 1 left: summed petition counts ---
    rows: 312,809   columns: 9
    distinct key: 153,165
    distinct fy: 7
    no missing values
--- BEFORE merge 1 right: dominant descriptors ---
    rows: 312,809   columns: 5
    distinct key: 153,165
    distinct fy: 7
    no missing values


--- AFTER merge 1: canonical employer-years ---
    rows: 312,809   columns: 12
    distinct key: 153,165
    distinct fy: 7
    no missing values
    rows: 312,809 -> 312,809

Petition conservation from raw USCIS rows to canonical employer-years:
    initial_approvals           757,806 ->    757,806  OK
    initial_denials             107,839 ->    107,839  OK
    continuing_approvals      1,884,861 ->  1,884,861  OK
    continuing_denials          122,194 ->    122,194  OK


Only employer-years with at least one initial adjudication have a defined initial-denial
outcome. Continuing-only cells remain in the history source but not in the modeling panel.

In [4]:
model_df = panel_all[panel_all.initial_total > 0].copy()
model_df["denial_rate"] = model_df.initial_denials / model_df.initial_total
model_df["any_denial"] = (model_df.initial_denials > 0).astype(int)
model_df["sector"] = model_df.naics.map(NAICS_LABEL).fillna("Unknown")

print(f"canonical employer-years with >=1 initial petition: {len(model_df):,}")
print(f"denial_rate mean={model_df.denial_rate.mean():.4f}, "
      f"median={model_df.denial_rate.median():.4f}")
print(f"exactly zero={(model_df.denial_rate == 0).mean():.1%}; "
      f"exactly one={(model_df.denial_rate == 1).mean():.1%}; "
      f"any denial={model_df.any_denial.mean():.1%}")

canonical employer-years with >=1 initial petition: 176,404
denial_rate mean=0.1281, median=0.0000
exactly zero=79.7%; exactly one=8.9%; any denial=20.3%


## 2. Strictly backward-looking sponsor history

### Merge 2 of 4: exact prior-fiscal-year activity

The lag table is stamped forward one year before joining. `prior_active_years` is the
number of earlier observed sponsor-years, calculated with a cumulative count. It never
uses future records.

In [5]:
hist = history_table(panel_all)
describe_frame(model_df, "BEFORE merge 2 left: outcome panel", keys=["key", "fy"])
describe_frame(hist, "BEFORE merge 2 right: prior-year history", keys=["key", "fy"])
before_rows = len(model_df)
model_df = model_df.merge(hist, on=["key", "fy"], how="left", validate="one_to_one")
describe_frame(model_df, "AFTER merge 2: prior-year history attached", keys=["key", "fy"])
print(f"    rows: {before_rows:,} -> {len(model_df):,}")
merge_report(model_df, "prev_petitions", weight_col="initial_total")

model_df["is_repeat_sponsor"] = model_df.prev_petitions.notna().astype(int)
model_df["prev_initial"] = model_df.prev_initial.fillna(0)
model_df["prev_denials"] = model_df.prev_denials.fillna(0)
model_df["prev_petitions"] = model_df.prev_petitions.fillna(0)
model_df["prev_denial_rate"] = model_df.prev_denial_rate.fillna(0)

panel_all = panel_all.sort_values(["key", "fy"])
prior_counts = panel_all.groupby("key").cumcount()
prior_lookup = pd.Series(
    prior_counts.to_numpy(),
    index=pd.MultiIndex.from_frame(panel_all[["key", "fy"]]))
model_index = pd.MultiIndex.from_frame(model_df[["key", "fy"]])
model_df["prior_active_years"] = prior_lookup.reindex(model_index).to_numpy()

model_df["log_initial"] = np.log1p(model_df.initial_total)
model_df["log_prev_initial"] = np.log1p(model_df.prev_initial)
model_df["continuing_share"] = (
    model_df.continuing_total / model_df.petitions_total.replace(0, np.nan)).fillna(0)
top_states = (model_df.groupby("state").initial_total.sum()
              .nlargest(12).index.tolist())
model_df["state_grp"] = np.where(model_df.state.isin(top_states),
                                  model_df.state, "OTHER")

model_df.to_csv(DATA / "analysis_panel.csv", index=False)
describe_frame(model_df, "USCIS analysis panel written to disk", keys=["key", "fy"])
assert not model_df.duplicated(["key", "fy"]).any()
print(f"    exact prior-year sponsors: {model_df.is_repeat_sponsor.mean():.1%}")
print(model_df[["denial_rate", "log_initial", "continuing_share",
                "prev_denial_rate", "prior_active_years"]].describe().round(3).to_string())

--- BEFORE merge 2 left: outcome panel ---
    rows: 176,404   columns: 15
    distinct key: 100,152
    distinct fy: 7
    no missing values
--- BEFORE merge 2 right: prior-year history ---
    rows: 312,809   columns: 6
    distinct key: 153,165
    distinct fy: 7
    columns with missing values: 1 (worst: prev_denial_rate at 43.6%)


--- AFTER merge 2: prior-year history attached ---
    rows: 176,404   columns: 19
    distinct key: 100,152
    distinct fy: 7
    columns with missing values: 4 (worst: prev_denial_rate at 65.6%)
    rows: 176,404 -> 176,404
    rows matched: 77,563 / 176,404  (44.0%)
    weighted by initial_total: 72.0%


--- USCIS analysis panel written to disk ---
    rows: 176,404   columns: 25
    distinct key: 100,152
    distinct fy: 7
    no missing values
    exact prior-year sponsors: 44.0%
       denial_rate  log_initial  continuing_share  prev_denial_rate  prior_active_years
count   176404.000   176404.000        176404.000        176404.000          176404.000
mean         0.128        1.118             0.282             0.045               1.208
std          0.303        0.729             0.325             0.173               1.568
min          0.000        0.693             0.000             0.000               0.000
25%          0.000        0.693             0.000             0.000               0.000
50%          0.000        0.693             0.000             0.000               1.000
75%          0.000        1.386             0.553             0.000               2.000
max          1.000        8.613             0.995             1.000               6.000


## 3. Clean and aggregate the DOL LCA records

Certified H-1B LCAs are retained, pay is annualized across units, and implausible annual
wages are removed before ratios are calculated. Wage levels I and II are aggregated with
the number of sponsored positions as weights. Employer-year wage medians remain
application-level medians because wages are reported per application.

### Merge 3 of 4: aggregate measures plus modal occupation

In [6]:
lca = load_lca()
describe_frame(lca, "cleaned certified H-1B LCA records", keys=["key", "fy"])

g = lca.groupby(["key", "fy"])
agg = g.agg(
    lca_filings=("key", "size"),
    lca_positions=("position_weight", "sum"),
    low_level_positions=("low_level_positions", "sum"),
    known_level_positions=("known_level_positions", "sum"),
    wage_ratio_med=("wage_ratio", "median"),
    wage_annual_med=("wage_annual", "median"),
    pw_annual_med=("pw_annual", "median"),
    share_full_time=("ft", "mean"),
    share_agent=("agent", "mean"),
    share_dependent=("dep", "mean"),
    share_willful=("willful", "mean"),
).reset_index()
agg["share_low_level"] = (
    agg.low_level_positions / agg.known_level_positions.replace(0, np.nan))

soc_weight = (lca.dropna(subset=["soc2"])
              .groupby(["key", "fy", "soc2"], as_index=False)
              .position_weight.sum())
modal_soc = (soc_weight.sort_values(
    ["key", "fy", "position_weight", "soc2"],
    ascending=[True, True, False, True])
    .drop_duplicates(["key", "fy"])
    [["key", "fy", "soc2"]]
    .rename(columns={"soc2": "soc2_modal"}))

describe_frame(agg, "BEFORE merge 3 left: LCA employer-year measures", keys=["key", "fy"])
describe_frame(modal_soc, "BEFORE merge 3 right: modal SOC", keys=["key", "fy"])
before_rows = len(agg)
agg = agg.merge(modal_soc, on=["key", "fy"], how="left", validate="one_to_one")
describe_frame(agg, "AFTER merge 3: complete LCA employer-years", keys=["key", "fy"])
print(f"    rows: {before_rows:,} -> {len(agg):,}")
assert not agg.duplicated(["key", "fy"]).any()

agg.to_csv(DATA / "lca_employer_year.csv", index=False)
print(agg.groupby("fy").agg(
    employer_years=("key", "size"), filings=("lca_filings", "sum"),
    positions=("lca_positions", "sum")).round().astype(int).to_string())

BEFORE filtering: 3,973,349 extracted rows


    after H-1B filter:      3,877,773


    after certified filter: 3,750,059


    wages inside [15,000, 2,000,000]: 99.2%


    blank employer keys dropped: 61


--- cleaned certified H-1B LCA records ---
    rows: 3,749,998   columns: 35
    distinct key: 150,390
    distinct fy: 6


    columns with missing values: 20 (worst: wage_to at 42.8%)


--- BEFORE merge 3 left: LCA employer-year measures ---
    rows: 329,829   columns: 14
    distinct key: 150,390
    distinct fy: 6
    columns with missing values: 5 (worst: share_low_level at 4.0%)
--- BEFORE merge 3 right: modal SOC ---
    rows: 291,693   columns: 3
    distinct key: 141,875
    distinct fy: 6
    no missing values


--- AFTER merge 3: complete LCA employer-years ---
    rows: 329,829   columns: 15
    distinct key: 150,390
    distinct fy: 6
    columns with missing values: 6 (worst: soc2_modal at 11.6%)
    rows: 329,829 -> 329,829


      employer_years  filings  positions
fy                                      
2017           60115   582462    1118998
2018           59229   611146    1237599
2019           58467   624683    1015220
2020           50119   550135     848293
2021           48782   784479    1735738
2022           53117   597093    1048973


## 4. Join USCIS outcomes to DOL filing characteristics

### Merge 4 of 4: exact canonical employer and fiscal year

The agencies publish no shared employer identifier. Both names therefore pass through
the documented `canon()` reduction in `code/src/names.py`. The left join retains every
USCIS outcome. A match means only that the fixed name key and fiscal year agree; it does
not imply petition-level linkage.

In [7]:
panel = pd.read_csv(DATA / "analysis_panel.csv", low_memory=False)
describe_frame(panel, "BEFORE merge 4 left: USCIS employer-years", keys=["key", "fy"])
describe_frame(agg, "BEFORE merge 4 right: DOL employer-years", keys=["key", "fy"])
print(f"    employer keys in common across any year: "
      f"{len(set(panel.key) & set(agg.key)):,}")
print(f"    canonicalizer check: {canon('COGNIZANT TECHNOLOGY SOLUTIONS U.S. CORPORATION')!r}")
print(f"                         {canon('COGNIZANT TECH SOLNS US CORP')!r}")

before_rows = len(panel)
merged = panel.merge(agg, on=["key", "fy"], how="left", validate="one_to_one")
describe_frame(merged, "AFTER merge 4: joined analysis panel", keys=["key", "fy"])
print(f"    rows: {before_rows:,} -> {len(merged):,}")
assert len(merged) == before_rows
assert not merged.duplicated(["key", "fy"]).any()

print("\nAll years:")
merge_report(merged, "lca_filings", weight_col="initial_total")
print("\nModeling window FY2017-FY2022:")
window = merged[merged.fy.isin(MODEL_YEARS)].copy()
merge_report(window, "lca_filings", weight_col="initial_total", by="fy")

match_year = window.groupby("fy").apply(
    lambda d: pd.Series({
        "employer_years": len(d),
        "matched_employer_years": d.lca_filings.notna().sum(),
        "matched_pct": d.lca_filings.notna().mean() * 100,
        "petition_weighted_pct": (d.loc[d.lca_filings.notna(), "initial_total"].sum()
                                  / d.initial_total.sum() * 100),
    }), include_groups=False)
match_year.to_csv(TABLES / "merge_match_by_year.csv")

--- BEFORE merge 4 left: USCIS employer-years ---
    rows: 176,404   columns: 25
    distinct key: 100,152
    distinct fy: 7
    no missing values
--- BEFORE merge 4 right: DOL employer-years ---
    rows: 329,829   columns: 15
    distinct key: 150,390
    distinct fy: 6
    columns with missing values: 6 (worst: soc2_modal at 11.6%)
    employer keys in common across any year: 76,811
    canonicalizer check: 'COGNIZANT TECH SOLNS'
                         'COGNIZANT TECH SOLNS'


--- AFTER merge 4: joined analysis panel ---
    rows: 176,404   columns: 38
    distinct key: 100,152
    distinct fy: 7
    columns with missing values: 13 (worst: soc2_modal at 39.3%)
    rows: 176,404 -> 176,404

All years:
    rows matched: 119,185 / 176,404  (67.6%)
    weighted by initial_total: 82.9%

Modeling window FY2017-FY2022:
    rows matched: 119,185 / 165,593  (72.0%)
    weighted by initial_total: 87.0%
         rows  matched_pct  weighted_pct
fy                                      
2017  22529.0         72.5          86.1
2018  27984.0         67.2          83.3
2019  33224.0         67.3          86.0
2020  27482.0         71.9          87.8
2021  25849.0         73.7          87.8
2022  28525.0         80.3          90.3


### Selection into the matched wage sample

Exact-name matching is imperfect and systematically easier for larger, established
sponsors. The model notebook therefore compares baseline and wage specifications on the
identical matched rows and reports this selection as a limitation.

In [8]:
w = window.copy()
w["size_decile"] = pd.qcut(
    w.initial_total.rank(method="first"), 10, labels=False) + 1
by_size = w.groupby("size_decile").apply(
    lambda d: pd.Series({
        "min_petitions": d.initial_total.min(),
        "max_petitions": d.initial_total.max(),
        "matched_pct": d.lca_filings.notna().mean() * 100,
        "petition_weighted_pct": (d.loc[d.lca_filings.notna(), "initial_total"].sum()
                                  / d.initial_total.sum() * 100),
    }), include_groups=False)
by_size.to_csv(TABLES / "merge_match_by_size.csv")
print(by_size.round(1).to_string())

unmatched = window[window.lca_filings.isna()].copy()
print("\nLargest unmatched canonical employers by initial petitions:")
print(unmatched.groupby("employer").initial_total.sum().nlargest(10).to_string())

present_any_year = unmatched.key.isin(set(agg.key))
print(f"\nUnmatched rows with the same key in another DOL year: {present_any_year.mean():.1%}")
print(f"Petition-weighted: "
      f"{unmatched.loc[present_any_year, 'initial_total'].sum() / unmatched.initial_total.sum():.1%}")

# Diagnostic only: use set membership, not an unreported data merge.
key_years = set(agg[["key", "fy"]].itertuples(index=False, name=None))
relaxed = np.array([
    pd.notna(f) or (k, y - 1) in key_years or (k, y + 1) in key_years
    for k, y, f in window[["key", "fy", "lca_filings"]].itertuples(index=False, name=None)
])
print(f"A +/-1-year diagnostic would cover {relaxed.mean():.1%} of employer-years and "
      f"{window.loc[relaxed, 'initial_total'].sum() / window.initial_total.sum():.1%} of petitions.")
print("It is not used because it changes the temporal meaning of the wage covariates.")

             min_petitions  max_petitions  matched_pct  petition_weighted_pct
size_decile                                                                  
1                      1.0            1.0         65.6                   65.6
2                      1.0            1.0         64.9                   64.9
3                      1.0            1.0         64.0                   64.0
4                      1.0            1.0         62.7                   62.7
5                      1.0            1.0         63.3                   63.3
6                      1.0            2.0         66.2                   66.9
7                      2.0            2.0         77.0                   77.0
8                      2.0            3.0         81.3                   81.9
9                      3.0            7.0         86.0                   86.2
10                     7.0         5500.0         88.8                   92.0

Largest unmatched canonical employers by initial petitions:
emp


Unmatched rows with the same key in another DOL year: 41.6%
Petition-weighted: 29.0%


A +/-1-year diagnostic would cover 82.7% of employer-years and 90.1% of petitions.
It is not used because it changes the temporal meaning of the wage covariates.


In [9]:
merged.to_csv(DATA / "analysis_panel_wages.csv", index=False)
describe_frame(merged, "FINAL panel written to disk", keys=["key", "fy"])

matched = merged.lca_filings.notna()
wage_cols = ["wage_ratio_med", "wage_annual_med", "pw_annual_med",
             "share_low_level", "share_agent", "share_dependent"]
print("\nWage block among exact employer-year matches:")
print(merged.loc[matched, wage_cols].describe().round(3).to_string())
print(f"\nUnique key-year assertion: {not merged.duplicated(['key', 'fy']).any()}")

--- FINAL panel written to disk ---
    rows: 176,404   columns: 38
    distinct key: 100,152
    distinct fy: 7
    columns with missing values: 13 (worst: soc2_modal at 39.3%)

Wage block among exact employer-year matches:
       wage_ratio_med  wage_annual_med  pw_annual_med  share_low_level  share_agent  share_dependent
count      118939.000       118939.000     118939.000       116973.000   118781.000       119185.000
mean            1.106        93821.044      83592.857            0.756        0.909            0.113
std             0.238        43874.201      29874.736            0.333        0.274            0.307
min             0.917        15600.000      15080.000            0.000        0.000            0.000
25%             1.000        68128.185      63320.080            0.526        1.000            0.000
50%             1.035        85842.000      80018.000            1.000        1.000            0.000
75%             1.130       106371.000      96595.000            1.0

The finished file contains exactly one row per canonical employer and fiscal year. The
next notebook describes the outcome and the selection into the wage subsample.